In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
pd.options.mode.chained_assignment = None

import pyfade

In [ ]:
PUMP = 'B-18 2'
SOURCE = 'C:/Users/Heitor/Documents/PosDoc/code/EPIC/Dados/merged_data.csv'
# EXAMPLE_SIGNAL = 'ESP motor temperature'
EXAMPLE_SIGNAL = None

In [ ]:
WAVELET = 'db2'
IGNORE_START = True
QUANTILE = 0.75
SUB_SIZE = 7*24
DIMENSION = 3
LEVELS = 5
USE_CUDA = True
SKIP_START=SUB_SIZE*10
LEFT_ONLY=True


mp_properties ={
    'quantile': QUANTILE,
    'skip_start': SKIP_START,
    'only_left': LEFT_ONLY,
    'use_cuda': USE_CUDA,
}

# 1 - Reading Signal

In [ ]:
# Load well data
if 'well_data' not in locals():
    data_full = pyfade.load_data(SOURCE)
    well_data = pyfade.get_well_data(data_full, PUMP, only_numerical=True, remove_ints=True, drop_na=True, drop_features = True)

shutdowns_all = pyfade.get_shutdowns(well_data)

In [ ]:
#fig, axs = pyfade.plot_multiple(well_data,style='.b',shutdowns=shutdowns_all);

In [ ]:
if EXAMPLE_SIGNAL is not None:
    sub_data = well_data[EXAMPLE_SIGNAL]
    shutdowns = [shutdowns_all[well_data.columns.get_loc(EXAMPLE_SIGNAL)]]
else:
    sub_data = well_data.copy()
    shutdowns = shutdowns_all

sub_data.interpolate(method='linear',inplace=True)
sub_data.dropna(inplace=True)


fig, axs = pyfade.plot_multiple(sub_data,style='xb',markersize=2,shutdowns=shutdowns);

# 2 - MP raw

In [ ]:
n_data = pyfade.znormalize(sub_data)

In [ ]:
MP = pyfade.get_MP(sub_data,SUB_SIZE,shutdowns=shutdowns,**mp_properties)


In [ ]:
fig, axs = pyfade.plot_multiple(sub_data,style='xb',shutdowns=None,markersize=2,height=3,width=10);
pyfade.plot_multiple(MP,fig=fig,axs=axs,style='r-',linewidth=2);

# 3 - Decomposition

In [ ]:
pyfade.get_signal_decomp(sub_data,wavelet=WAVELET, create_plot=True, level = LEVELS, height= 2, width= 10,markersize=2,style='xb');

# 4 - MP wavelet

In [ ]:
MP_wave, _ =  pyfade.get_MP_from_wavelets(sub_data,SUB_SIZE,LEVELS, wavelet=WAVELET,on_signals=True,create_plot=True,plot_data=True,shutdowns=shutdowns,height=2, width=10,markersize=2,style=['r-','xb'],**mp_properties)

In [ ]:
KDP = pyfade.get_KDP(sub_data,SUB_SIZE,pre_calc_MP=MP_wave,shutdowns=shutdowns)
fig, axs = pyfade.plot_multiple(KDP,same_limits=False,style='r-',height=2, width=10);

In [ ]:
#sub_data.drop('ESP discharge temperature sensor',inplace=True,axis=1)
MPs = pyfade.wavelet_MP_from_KDP(sub_data,SUB_SIZE,DIMENSION,level=LEVELS,wavelet=WAVELET, on_signals=True,shutdowns=shutdowns, **mp_properties)

In [ ]:
no_outlier = sub_data.copy()
a = no_outlier.quantile(0.95,axis=0)
no_outlier[no_outlier>a] = np.nan
a = no_outlier.quantile(0.2,axis=0)
no_outlier[no_outlier<a] = np.nan

ind = np.where([col in ['VSD power frequency', 'Well head Temperature'] for col in no_outlier.columns])[0].astype(int)


fig, axs = pyfade.plot_multiple(no_outlier.iloc[:,ind],style='xb',markersize=2,height=2, width=10,shutdowns=None);
pyfade.plot_multiple([MPs[i] for i in ind],fig=fig,axs=axs,style='r-',linewidth=2,ylabel='Matrix Profile');

In [ ]:
fig.savefig('All_MP.png', format='png', dpi=300)

In [ ]:
KDP = pyfade.get_KDP(sub_data,SUB_SIZE,pre_calc_MP=MPs,shutdowns=shutdowns,**mp_properties)
fig, axs = pyfade.plot_multiple(KDP,same_limits=False,height=1, width=10,linewidth=2,style='r-');

In [ ]:
fig.savefig('All_KDP.png', format='png', dpi=300)